# Regressão Softmax com dados do MNIST utilizando gradiente descendente estocástico por minibatches

O objetivo deste notebook é ilustrar
- o uso do gradiente estocástico por mini-batchs
- utilizando as classes Dataset e DataLoater.

A apresentação da perda nos gráficos é um pouco diferente da usual, mostrando a perda de cada um dos vários minibatches dentro de cada época, de forma que as épocas são apresentadas com valores fracionários.

## Importação das bibliotecas

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.autograd import Variable
from torch.utils.data import DataLoader

import torchvision
from torchvision.datasets import MNIST

## Dataset e dataloader

### Definição do tamanho do minibatch

In [ ]:
# batch_size = 100
batch_sizes = [50, 75, 100]

### Carregamento, criação dataset e do dataloader

In [ ]:
dataset_dir = 'MNIST/'
batch_trainings = {}

# Load the full MNIST training dataset
full_train_dataset = MNIST(dataset_dir, train=True, download=True,
                      transform=torchvision.transforms.ToTensor())

# Define the split ratio (e.g., 80% for training, 20% for validation)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

# Split the dataset
dataset_train, dataset_val = torch.utils.data.random_split(full_train_dataset, [train_size, val_size])

# Create DataLoaders for training and validation
for size in batch_sizes:
     batch_trainings[size] = {
         'train': DataLoader(dataset_train, batch_size=size, shuffle=True),
         'val': DataLoader(dataset_val, batch_size=size, shuffle=False)
     }
# loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
# loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False) # No need to shuffle validation data

print(f'Número total de amostras de treinamento original: {len(full_train_dataset)}')
print(f'Número de amostras de treinamento: {len(dataset_train)}')
print(f'Número de amostras de validação: {len(dataset_val)}')
# print('Número de minibatches de treinamento:', len(loader_train))
# print('Número de minibatches de validação:', len(loader_val))
for size in batch_sizes:
    print(f'Número de minibatches de treinamento (batch size {size}):', len(batch_trainings[size]['train']))
    print(f'Número de minibatches de validação (batch size {size}):', len(batch_trainings[size]['val']))

    x_train_sample, y_train_sample = next(iter(batch_trainings[size]['train']))
    batch_trainings[size].update({"x_train_sample":x_train_sample, "y_train_sample":y_train_sample})
    print("\nDimensões dos dados de um minibatch de treinamento:", x_train_sample.size())
    print("Valores mínimo e máximo dos pixels: ", torch.min(x_train_sample), torch.max(x_train_sample))
    print("Tipo dos dados das imagens:         ", type(x_train_sample))
    print("Tipo das classes das imagens:       ", type(y_train_sample))

### Usando todas as amostras do MNIST

Neste exemplo utilizaremos todas as amostras de treinamento.

In [ ]:
# The original `total_samples` was for the full training set. Now we have a split.
for size in batch_sizes:
    n_batches_train = len(batch_trainings[size]['train'])
    n_batches_val = len(batch_trainings[size]['val'])
    total_train_samples = len(dataset_train)
    total_val_samples = len(dataset_val)

    print(f'Número de minibatches de treinamento (batch size {size}): {n_batches_train}')
    print(f'Número de minibatches de validação (batch size {size}): {n_batches_val}')
    print(f'Total de amostras de treinamento: {total_train_samples}')
    print(f'Total de amostras de validação: {total_val_samples}')

## Modelo

In [ ]:
model = torch.nn.Linear(28*28, 10)

In [ ]:
x = torch.ones(28*28).reshape(1, 784)
print(x.shape)
predict = model(x)
predict

## Treinamento

Durante o treinamento, acompanharemos a perda e a acurácia para os conjuntos de treinamento e validação em cada época.

### Inicialização dos parâmetros

In [ ]:
n_epochs = 5
learningRate = 0.5

# Utilizaremos CrossEntropyLoss como função de perda
criterion = torch.nn.CrossEntropyLoss()

# Gradiente descendente
optimizer = torch.optim.SGD(model.parameters(), lr=learningRate)

### Laço de treinamento dos parâmetros

In [ ]:
# train_losses_per_epoch = []
# val_losses_per_epoch = []
# train_accuracies_per_epoch = []
# val_accuracies_per_epoch = []

loop_train_sizes = {}
for size in batch_sizes:
    trainnings = {
        "train_losses_per_epoch": [],
        "val_losses_per_epoch": [],
        "train_accuracies_per_epoch": [],
        "val_accuracies_per_epoch": []
        }
    loop_train_sizes[size] = trainnings
    loader_train = batch_trainings[size]['train']
    loader_val = batch_trainings[size]['val']

    for epoch in range(n_epochs):
        # Training phase
        model.train() # Set model to training mode
        correct_train = 0
        total_train = 0
        running_train_loss = 0.0

        for i, (x_train, y_train) in enumerate(loader_train):
            inputs = Variable(x_train.view(-1, 28 * 28))
            labels = Variable(y_train)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            print(
                f'Epoch: {epoch+1}/{n_epochs}, '
                f'batch: {i+1}/{len(loader_train)}'
            )

        avg_train_loss = running_train_loss / len(loader_train)
        train_accuracy = correct_train / total_train
        loop_train_sizes[size]["train_losses_per_epoch"].append(avg_train_loss)
        loop_train_sizes[size]["train_accuracies_per_epoch"].append(train_accuracy)

        # Validation phase
        model.eval() # Set model to evaluation mode
        correct_val = 0
        total_val = 0
        running_val_loss = 0.0

        with torch.no_grad(): # Disable gradient calculations during validation
            for x_val, y_val in loader_val:
                inputs_val = Variable(x_val.view(-1, 28 * 28))
                labels_val = Variable(y_val)

                outputs_val = model(inputs_val)
                loss_val = criterion(outputs_val, labels_val)

                running_val_loss += loss_val.item()
                _, predicted_val = torch.max(outputs_val.data, 1)
                total_val += labels_val.size(0)
                correct_val += (predicted_val == labels_val).sum().item()

        avg_val_loss = running_val_loss / len(loader_val)
        val_accuracy = correct_val / total_val
        loop_train_sizes[size]["val_losses_per_epoch"].append(avg_val_loss)
        loop_train_sizes[size]["val_accuracies_per_epoch"].append(val_accuracy)

        print(f'Epoch [{epoch+1}/{n_epochs}], ' \
            f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.4f}, ' \
            f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.4f}')

In [ ]:
print('--- Treinamento Finalizado ---')
for size in batch_sizes:
    print(f'\nResultados para batch size {size}:')
    print(f'Última perda de treinamento: {loop_train_sizes[size]["train_losses_per_epoch"][-1]:.4f}')
    print(f'Última acurácia de treinamento: {loop_train_sizes[size]["train_accuracies_per_epoch"][-1]:.4f}')
    print(f'Última perda de validação: {loop_train_sizes[size]["val_losses_per_epoch"][-1]:.4f}')
    print(f'Última acurácia de validação: {loop_train_sizes[size]["val_accuracies_per_epoch"][-1]:.4f}')

# print(f'Última perda de treinamento: {train_losses_per_epoch[-1]:.4f}')
# print(f'Última acurácia de treinamento: {train_accuracies_per_epoch[-1]:.4f}')
# print(f'Última perda de validação: {val_losses_per_epoch[-1]:.4f}')
# print(f'Última acurácia de validação: {val_accuracies_per_epoch[-1]:.4f}')

### Visualizando a evolução da perda e acurácia

In [ ]:
for size in batch_sizes:
    train_losses_per_epoch = loop_train_sizes[size]["train_losses_per_epoch"]
    val_losses_per_epoch = loop_train_sizes[size]["val_losses_per_epoch"]
    train_accuracies_per_epoch = loop_train_sizes[size]["train_accuracies_per_epoch"]
    val_accuracies_per_epoch = loop_train_sizes[size]["val_accuracies_per_epoch"]

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, n_epochs + 1), train_losses_per_epoch, label='Perda de Treinamento')
    plt.plot(range(1, n_epochs + 1), val_losses_per_epoch, label='Perda de Validação')
    plt.xlabel('Época')
    plt.ylabel('Perda')
    plt.title('Evolução da Perda (Treinamento vs. Validação) - Batch Size ' + str(size))
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, n_epochs + 1), train_accuracies_per_epoch, label='Acurácia de Treinamento')
    plt.plot(range(1, n_epochs + 1), val_accuracies_per_epoch, label='Acurácia de Validação')
    plt.xlabel('Época')
    plt.ylabel('Acurácia')
    plt.title('Evolução da Acurácia (Treinamento vs. Validação) - Batch Size ' + str(size))
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

### Observações sobre os gráficos

Estes gráficos mostram a perda e a acurácia calculadas ao final de cada época para os conjuntos de treinamento e validação. Idealmente, a perda de treinamento e validação deve diminuir, e a acurácia deve aumentar, com as curvas de validação seguindo de perto as de treinamento. Divergências significativas podem indicar overfitting ou underfitting.

# Atividades

## Perguntas

1. Qual é o tamanho do mini-batch?
2. Em uma época, quantos mini-batches existem?
3. Qual é a definição de época?

### Respostas

1) O tamanho do mini-batch édefinido pelo parâmetro batch_size do DataLoader, no caso sendo definido no inicio do notebook com valor 100.
2) Para obtermos o quantos mini-batches existem, devemos entender que ele é calculado pelo resultado de `mini-batches = total_amostras/batch_size`,no caso, como temos 6000 amostras, temos então 600 mini-batches.
3) É um ciclo completo de treinamento, quando a rede processa todo o conjunto de dados.

## Exercícios


1. Coloque um print no final de cada minibatch, no mesmo estilo do print do final de época, no seguinte estilo:
    - Época: 1/4, batch: 600
2. Altere o tamanho de minibatch (batch_size) algumas vezes, refaça o treinamento, e compare no gráfico abaixo a queda da perda para cada tamanho de minibatch.

## Conclusões sobre os experimentos deste notebook
